# 7.1 · 回归评估指标 / Regression Metrics

> **课程定位 / Where this fits**
> 第 1 课，**Part 7 · 模型评估与优化**。
> Lesson 1, **Part 7 · Model Evaluation & Tuning**.
>
> Part 4 建了一堆回归模型，但"哪个更好"取决于**用什么指标衡量**。这一课系统讲回归指标：MAE/MSE/RMSE/R²/调整 R²/MAPE/sMAPE——它们各自惩罚什么、何时用哪个、有什么坑。**选错指标比选错模型更致命**，因为它会让你一路优化错方向。
> Part 4 built many regression models, but "which is better" depends on **what metric you measure with**. This lesson systematically covers regression metrics: MAE/MSE/RMSE/R²/adjusted R²/MAPE/sMAPE — what each penalizes, when to use which, and their pitfalls. **Choosing the wrong metric is worse than the wrong model** — it steers all your optimization off-target.
>
> 💼 **实战/面试视角**："MAE 和 RMSE 区别 / 什么时候用哪个 / R² 能为负吗" 是回归岗高频题。
> 💼 **Practical/interview angle:** "MAE vs RMSE / when to use which / can R² be negative" are common regression questions.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $y_i, \hat y_i$ —— 真实值、预测值 / actual, predicted
> - $\bar y$ —— 真实值的均值 / mean of actuals
> - $n$ 样本数, $d$ 特征数 / samples, features

> 💡 **面试相关 / Interview-relevant**
> - "MAE vs MSE/RMSE 的区别（对大误差/异常值）"（出镜率 ★★★★★）
> - "R² 的含义 / 能不能为负"（★★★★★）
> - "R² vs 调整 R²"（★★★★）
> - "MAPE 的缺陷 / 什么时候不能用"（★★★★）
> - "指标怎么对应损失函数 / 业务成本"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 MAE / MSE / RMSE 各惩罚什么、对异常值的敏感度。
   Understand what MAE / MSE / RMSE penalize and their outlier sensitivity.
2. 理解 R² 的含义、它**可以为负**、以及调整 R²。
   Understand R²'s meaning, that it **can be negative**, and adjusted R².
3. 掌握 MAPE / sMAPE 的相对误差视角及其缺陷。
   Master MAPE / sMAPE (relative error) and their pitfalls.
4. 学会**按业务成本选指标**。
   Learn to **pick the metric by business cost**.

## 目录 / TOC
1. [先建直觉：误差怎么量 ⭐](#1)
2. [🏠 数据 + MAE/MSE/RMSE ⭐](#2)
3. [异常值敏感度对比 ⭐](#3)
4. [R² 与调整 R² ⭐](#4)
5. [MAPE / sMAPE：相对误差 ⭐](#5)
6. [按业务选指标 + 小结](#6)


 <a id="1"></a>
## 1. 先建直觉：误差怎么量 ⭐ / Intuition: Measuring Error

回归指标都在回答同一个问题：**预测离真实有多远？** 区别在于"怎么把一堆误差汇总成一个数"，而这个选择决定了模型会**讨好谁、忽略谁**。
All regression metrics answer one question: **how far are predictions from the truth?** The difference is "how to aggregate many errors into one number", and that choice decides **who the model pleases and who it ignores**.

三种最基本的汇总方式：
Three fundamental aggregations:
- **取绝对值再平均（MAE）**：每个误差等权，所有点同等重要。
  **Average the absolute values (MAE):** every error weighted equally, all points matter the same.
- **取平方再平均（MSE/RMSE）**：误差先平方，**大误差被放大**，模型会优先消灭大误差。
  **Average the squares (MSE/RMSE):** square first, so **large errors are amplified**; the model prioritizes killing big errors.
- **看相对误差（MAPE）**：误差除以真实值，关注"错了百分之几"而非绝对量。
  **Relative error (MAPE):** divide by the actual, focusing on "percent wrong" rather than absolute amount.

没有"最好的指标"，只有"最匹配业务的指标"——这是本课的核心。
There is no "best metric", only the one that best matches the business — the theme of this lesson.


<a id="2"></a>
## 2. 数据 + MAE/MSE/RMSE ⭐ / Data & MAE/MSE/RMSE

用 **California Housing**（4.1 用过）训一个回归模型，逐个算指标。三个绝对误差指标：
Using **California Housing** (from 4.1), train a regressor and compute metrics. The three absolute-error metrics:

$$\text{MAE} = \frac1n\sum|y_i-\hat y_i|, \quad \text{MSE} = \frac1n\sum(y_i-\hat y_i)^2, \quad \text{RMSE} = \sqrt{\text{MSE}}$$

- **MAE**：和目标同单位，直观（"平均差 X 万美元"），对异常值稳健。
  **MAE:** same unit as the target, intuitive ("off by X on average"), robust to outliers.
- **MSE**：单位是目标的平方（不直观），但数学上好（可导，是 OLS 的损失）。
  **MSE:** unit is the target squared (not intuitive), but mathematically nice (differentiable, OLS's loss).
- **RMSE**：开根回到目标单位，同时**保留了对大误差的重罚**——回归报告里最常用。
  **RMSE:** square-root back to the target's unit while **keeping the heavy penalty on large errors** — the most reported regression metric.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)

data = fetch_california_housing(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, random_state=0)
model = LinearRegression().fit(X_tr, y_tr)
y_pred = model.predict(X_te)

mae = mean_absolute_error(y_te, y_pred)
mse = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)                                # = mean_squared_error 后开根
print(f"MAE  = {mae:.4f} (10万$, 同单位, 直观)")
print(f"MSE  = {mse:.4f} (单位是 '10万$'², 不直观)")
print(f"RMSE = {rmse:.4f} (10万$, 同单位 + 重罚大误差)")
print(f"\nRMSE > MAE 恒成立; 差距 {rmse-mae:.3f} 越大 → 大误差越多(RMSE 对它们更敏感)")


<a id="3"></a>
## 3. 异常值敏感度对比 ⭐ / Outlier Sensitivity

**MAE 和 RMSE 最核心的区别就是对异常值的敏感度**（面试必考）。因为 RMSE 先平方，一个巨大的误差会被放大、主导整个指标；MAE 则对每个误差一视同仁。下面在预测里**人为注入一个大错误**，看两个指标各跳多少。
**The core difference between MAE and RMSE is outlier sensitivity** (always asked). Because RMSE squares first, one huge error gets amplified and dominates the metric; MAE treats every error equally. Below we **inject one big error** into the predictions and watch how much each metric jumps.


In [ ]:
y_pred_clean = y_pred.copy()
y_pred_bad = y_pred.copy()
y_pred_bad[0] += 20          # 在第 0 个预测上制造一个巨大误差 / one giant error

print(f"{'':20} {'MAE':>10} {'RMSE':>10}")
for name, p in [("正常预测 clean", y_pred_clean), ("加 1 个大误差 +1 outlier", y_pred_bad)]:
    print(f"{name:20} {mean_absolute_error(y_te, p):>10.4f} {np.sqrt(mean_squared_error(y_te, p)):>10.4f}")

# 计算各自的相对涨幅 / relative jump
mae_jump = mean_absolute_error(y_te, y_pred_bad)/mean_absolute_error(y_te, y_pred_clean) - 1
rmse_jump = np.sqrt(mean_squared_error(y_te, y_pred_bad))/np.sqrt(mean_squared_error(y_te, y_pred_clean)) - 1
print(f"\n仅 1 个大误差导致: MAE 涨 {mae_jump:.0%}, RMSE 涨 {rmse_jump:.0%}")
print("→ RMSE 对异常值/大误差远比 MAE 敏感(平方放大)")
print("选择: 在意'最坏情况/大误差' → RMSE; 数据有异常值且想稳健 → MAE")


<a id="4"></a>
## 4. R² 与调整 R² ⭐ / R² and Adjusted R²

MAE/RMSE 是有量纲的（依赖目标大小），不好跨问题比较。**R²（决定系数）**给出一个无量纲的"解释了多少"：
MAE/RMSE have units (depend on target scale), hard to compare across problems. **R² (coefficient of determination)** gives a unitless "how much is explained":

$$R^2 = 1 - \frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2} = 1 - \frac{\text{模型的误差}}{\text{'永远预测均值'的误差}}$$

- **R²=1**：完美；**R²=0**：和"永远预测均值"一样烂；**R²<0**：比预测均值还差（面试常问"R² 能为负吗"——**能**，模型在测试集上很糟时就会）。
  **R²=1:** perfect; **R²=0:** no better than always predicting the mean; **R²<0:** worse than the mean (a common question — yes, R² can be negative when the model is bad on test data).
- **调整 R²**：R² 永远随特征增多而升（哪怕加噪声列），**调整 R² 惩罚特征数**，只有真正有用的特征才让它上升——比较不同特征数的模型时用它。
  **Adjusted R²:** R² always rises with more features (even noise); adjusted R² **penalizes feature count**, rising only for useful features — use it to compare models with different feature counts.


In [ ]:
print(f"model R² = {r2_score(y_te, y_pred):.4f}  (解释了 {r2_score(y_te, y_pred):.0%} 的目标方差)\n")

# 演示 R² 可以为负: 一个故意很差的"模型"(永远预测一个偏离的常数) / R² can be negative
bad_const = np.full_like(y_te, y_tr.mean() + 3)   # 永远预测 "均值+3"(明显偏)
print(f"故意很差的常数预测 R² = {r2_score(y_te, bad_const):.3f}  ← 负数! 比预测均值还差")
print(f"永远预测训练均值 R² = {r2_score(y_te, np.full_like(y_te, y_tr.mean())):.3f}  ≈ 0 (R² 的基准)\n")

# 调整 R²: 加噪声特征看 R² vs adj R² / adjusted R² penalizes useless features
def adjusted_r2(r2, n, d):
    return 1 - (1-r2)*(n-1)/(n-d-1)        # n=样本数, d=特征数

rng = np.random.default_rng(0)
Xtr_noise = np.c_[X_tr.values, rng.normal(size=(len(X_tr), 50))]   # 加 50 个纯噪声特征
Xte_noise = np.c_[X_te.values, rng.normal(size=(len(X_te), 50))]
m2 = LinearRegression().fit(Xtr_noise, y_tr)
r2_noise = r2_score(y_te, m2.predict(Xte_noise))
print(f"加 50 个噪声特征后: R² = {r2_noise:.4f} (训练 R² 几乎只升不降), "
      f"调整 R² = {adjusted_r2(r2_noise, len(X_te), Xte_noise.shape[1]):.4f}")
print("→ 调整 R² 因噪声特征而下降 → 它能识破'靠堆特征刷 R²'的把戏")


<a id="5"></a>
## 5. MAPE / sMAPE：相对误差 ⭐ / Relative Error

有时我们关心的是"**错了百分之几**"而非绝对量——预测 100 万差 10 万（10%）和预测 10 万差 10 万（100%）严重程度完全不同。**MAPE（平均绝对百分比误差）** $=\frac1n\sum\frac{|y_i-\hat y_i|}{|y_i|}$ 衡量相对误差。
Sometimes we care about "**percent wrong**" not absolute amount — being off by 10 on a true 100 (10%) differs hugely from off by 10 on a true 10 (100%). **MAPE** $=\frac1n\sum\frac{|y_i-\hat y_i|}{|y_i|}$ measures relative error.

**MAPE 的两个致命坑**（面试要点）：(1) 真实值接近 0 时分母爆炸 → MAPE 飙到无穷；(2) **不对称**——它对"低估"和"高估"罚得不一样，会诱导模型系统性偏低预测。**sMAPE（对称 MAPE）**用 $\frac{|y-\hat y|}{(|y|+|\hat y|)/2}$ 缓解不对称，但仍怕 0 附近。
**MAPE's two fatal pitfalls** (interview points): (1) the denominator blows up near zero → MAPE shoots to infinity; (2) it's **asymmetric** — it penalizes under- and over-estimation differently, biasing the model to under-predict. **sMAPE** uses $\frac{|y-\hat y|}{(|y|+|\hat y|)/2}$ to ease the asymmetry, but still struggles near zero.


In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

# California 目标全为正且不接近0, MAPE 可用 / safe here (target positive, away from 0)
mape = mean_absolute_percentage_error(y_te, y_pred)
print(f"MAPE = {mape:.1%}  (平均相对误差约 {mape:.0%})")

def smape(y, yhat):
    return np.mean(np.abs(y-yhat) / ((np.abs(y)+np.abs(yhat))/2))
print(f"sMAPE = {smape(y_te.values, y_pred):.1%}\n")

# 演示 MAPE 在 0 附近爆炸 / MAPE explodes near zero
y_small = np.array([0.01, 0.5, 1.0, 2.0])
y_hat   = np.array([0.10, 0.5, 1.0, 2.0])      # 只有第一个(真值0.01)预测差一点
print("真值含接近 0 的样本时:")
print(f"  真实 {y_small}, 预测 {y_hat}")
print(f"  MAPE = {mean_absolute_percentage_error(y_small, y_hat):.0%}  ← 被那个 0.01 的样本拉爆!")
print("  → 目标含 0 或接近 0 时禁用 MAPE; 用 MAE/RMSE 或对数误差")


<a id="6"></a>
## 6. 按业务选指标 + 小结 / Pick by Business & Summary

**没有万能指标，要按业务成本选**（这是最重要的实战意识）：
**No universal metric; pick by business cost** (the key practical mindset):

| 业务情形 / Situation | 选哪个 / Use |
|---|---|
| 大误差代价特别高（如电网负荷预测）| **RMSE**（重罚大误差）|
| 有异常值、想要稳健、误差等权 | **MAE** |
| 关心相对误差、目标跨数量级（销量预测）| **MAPE / sMAPE**（注意 0 附近）|
| 跨问题/跨数据集比较、汇报"解释力" | **R²**（无量纲）|
| 模型选择、不同特征数比较 | **调整 R²** |

> ⚠️ **关键**：**训练用的损失**和**评估用的指标**可以不同，但要协调。比如想优化 MAE 就别用平方损失训练——用分位数/绝对损失(4.14)。
> ⚠️ **Key:** the **training loss** and the **evaluation metric** can differ but should align. To optimize MAE, don't train with squared loss — use absolute/quantile loss (4.14).

```
MAE: 同单位, 等权, 抗异常; MSE: 平方(放大大误差, OLS损失); RMSE: 开根回单位+重罚大误差
RMSE vs MAE 核心区别 = 对大误差/异常值的敏感度(RMSE 敏感)
R²: 1-模型误差/均值误差; 可为负(比均值还差); 调整 R² 惩罚特征数(防堆特征刷分)
MAPE: 相对误差%; 坑=0附近爆炸+不对称(诱导低估); sMAPE 缓解但仍怕0
按业务成本选指标; 训练损失与评估指标要协调
```

### 💡 面试速查 / Interview cheat-sheet
1. **RMSE 重罚大误差/对异常值敏感, MAE 等权/稳健**。
   RMSE penalizes large errors / outlier-sensitive; MAE is equal-weight / robust.
2. **R² 可以为负**(模型比预测均值还差); 0=均值基准, 1=完美。
   R² can be negative (worse than the mean); 0 = mean baseline, 1 = perfect.
3. **调整 R²** 惩罚特征数, 选模型用它(R² 只升不降)。
   Adjusted R² penalizes feature count; use it for model selection.
4. **MAPE 在 0 附近爆炸且不对称**(诱导低估), 目标含 0 时禁用。
   MAPE explodes near 0 and is asymmetric (biases under-prediction); avoid when target has zeros.
5. **按业务成本选指标**; 训练损失与评估指标要协调。
   Pick the metric by business cost; align training loss with the eval metric.

### 下一节 / Next
**7.2 分类评估指标**——分类的对应版: 混淆矩阵衍生的 P/R/F1、ROC-AUC/PR-AUC、MCC、对数损失, 以及多分类的 micro/macro 平均。
**7.2 Classification Metrics** — the classification counterpart: P/R/F1 from the confusion matrix, ROC-AUC/PR-AUC, MCC, log loss, and micro/macro averaging.
